# 04 · Crop-health classes, zones and weak vegetation

The crop-health engine turns an index raster into explainable classes.
Thresholds are **never hard-coded**: they come from `config.yaml` and every
result reports the breakpoints that were used.

| class | range | meaning |
|---|---|---|
| Very low vegetation | < 0.20 | bare soil, water, severe stress |
| Low vegetation | 0.20 – 0.40 | sparse or emerging canopy |
| Moderate vegetation | 0.40 – 0.60 | partial ground cover |
| Healthy vegetation | 0.60 – 0.80 | closed, vigorous canopy |
| Very healthy vegetation | > 0.80 | peak vigour |

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## Prepare the NDVI raster

Same clip + index path as notebook 03, in one cell.

In [ ]:
metadata = data.get_raster_metadata(RASTER)
resolution = indices.resolve_band_map(metadata, None)
aoi = data.load_aoi(AOI_FILE)

clipped = geospatial.clip_raster_to_aoi(
    str(RASTER), aoi["gdf"], bands=sorted(set(resolution["bands"].values())), max_pixels=config.max_analysis_pixels()
)
stack = preprocessing.band_stack(clipped, resolution["bands"])
valid = np.asarray(stack["valid_mask"])

ndvi = np.asarray(indices.compute_index("NDVI", stack["bands"], valid=valid)["array"], dtype="float32")
statistics = indices.index_statistics(ndvi, valid=valid)
print("mean NDVI: %.3f over %d valid pixels" % (statistics["mean"], statistics["valid_pixels"]))

## 1. Thresholds come from configuration

`config.yaml` is the single source of truth; any run can override it without editing code.

In [ ]:
thresholds = config.health_thresholds()
print("configured in config.yaml:", json.dumps(thresholds))
print("defaults / env overrides resolved by config.py:", json.dumps(config.get_config()["crop_health"]["thresholds"]))

## 2. Classify every pixel and summarise the field

In [ ]:
classes = analysis.classify_health(ndvi, thresholds=thresholds, valid=valid)
health = analysis.health_statistics(
    classes,
    pixel_area_ha=stack["pixel_area_ha"],
    thresholds=thresholds,
    field_area_ha=aoi["area_ha"],
)
print("analysed area (ha):", round(health["analysed_area_ha"], 2))
print("healthy %:", health["healthy_pct"], "| weak %:", health["weak_pct"])
pd.DataFrame(health["classes"])[["key", "label", "range", "pixels", "percentage", "area_ha", "severity"]]

## 3. Health zones: where the classes actually are

Classes become contiguous polygons with real areas and centroids - the basis for scouting maps and, later, for training labels.

In [ ]:
zones = analysis.health_zones(
    classes,
    stack["transform"],
    stack["crs"],
    min_area_ha=0.05,
    pixel_area_ha=stack["pixel_area_ha"],
)
print("total zones:", zones["total_zones"], "| per class:", zones["zone_count"])

rows = []
for group in zones["zones"].values():
    for zone in group:
        rows.append({
            "class": zone["label"],
            "area_ha": zone["area_ha"],
            "centroid_lonlat": [round(value, 4) for value in zone["centroid_lonlat"]],
        })
print(f"{len(rows)} zones, largest first:")
pd.DataFrame(sorted(rows, key=lambda row: row["area_ha"], reverse=True)[:8])

## 4. Weak vegetation patches

Low-vigour areas are grouped into patches so the output is actionable ('12 ha in three patches') instead of a noisy per-pixel list.

In [ ]:
weak = analysis.detect_weak_vegetation(
    ndvi,
    stack["transform"],
    stack["crs"],
    thresholds=thresholds,
    valid=valid,
    min_area_ha=0.05,
    pixel_area_ha=stack["pixel_area_ha"],
)
print("weak threshold :", weak["threshold"])
print("patches        :", weak["patch_count"], "| largest (ha):", weak["largest_patch_ha"])
print("weak area (ha) :", weak.get("weak_area_ha", "n/a"))

patches = pd.DataFrame(weak["patches"])
patches.head(8) if len(patches) else print("No weak patch above the minimum area.")

## 5. Statistical anomalies

Outliers relative to the field itself (z-score or IQR) flag spots that differ from their own surroundings, independently of the absolute thresholds.

In [ ]:
anomalies = analysis.detect_anomalies(ndvi, valid=valid, method="zscore", threshold=2.0)
print({key: anomalies[key] for key in ("method", "threshold", "count", "percentage") if key in anomalies})

## 6. Field summary and plain-language interpretation

The interpretation is generated from the numbers above - and it is explicit about what NDVI cannot conclude.

In [ ]:
summary = analysis.field_summary(
    index_name="NDVI",
    statistics=statistics,
    health=health,
    weak_vegetation=weak,
    anomalies=anomalies,
    field_area_ha=aoi["area_ha"],
    analysed_area_ha=health["analysed_area_ha"],
    thresholds=thresholds,
)
summary["largest_patch_ha"] = weak.get("largest_patch_ha")

interpretation = analysis.interpret(summary)
print("VERDICT    :", interpretation["verdict"])
print("CONFIDENCE :", interpretation["confidence"])
for point in interpretation["points"]:
    print(" -", point)
print("\nDISCLAIMER :", interpretation["disclaimer"])

## 7. The full visual bundle

One call produces the health map, the class-distribution chart, the weak-vegetation map and the NDVI map with field boundaries burned in.

In [ ]:
figure_dir = NOTEBOOK_OUT / "04_crop_health"
figure_dir.mkdir(parents=True, exist_ok=True)

assets = visualization.render_result_bundle(
    figure_dir,
    rgb=preprocessing.composite_rgb({key: stack["bands"][key] for key in ("red", "green", "blue")}, valid),
    index_array=ndvi,
    classes=classes,
    weak_mask=np.asarray(weak["mask"]) if weak["mask"].any() else None,
    weak_threshold=weak["threshold"],
    weak_patch_count=weak["patch_count"],
    statistics=statistics,
    health=health,
    transform=stack["transform"],
    crs=stack["crs"],
    aoi_gdf=aoi["gdf"],
    bounds=clipped["clip_bounds"],
    valid=valid,
    index_name="NDVI",
    dataset_label=RASTER.name,
    extra_caption="AOI %s · %d valid pixels" % (aoi["name"], statistics["valid_pixels"]),
)
for key, item in assets.items():
    print(f"  {key:22s} {item['kind']:8s} {pathlib.Path(item['path']).relative_to(ROOT)}")

In [ ]:
health_map = pathlib.Path(assets["health_map"]["path"])
figure, axes = plt.subplots(figsize=(7, 5.5))
axes.imshow(plt.imread(health_map))
axes.axis("off")
axes.set_title("Crop-health map (from the visualization engine)")
plt.tight_layout()
plt.show()

**Reading the result.** The share of pixels per class, the weakest patches
and the anomaly count together say *where to look*. They do not say *why* -
nutrient stress, waterlogging, pests and disease all look alike in a single
NDVI snapshot. Notebook 07 turns a stack of dates into a trajectory, which is
where genuine change becomes visible.